<a href="https://colab.research.google.com/github/Eliascc5/English_proficiency_prediction_NLP/blob/main/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training and Evaluation

**Goal:** train a neural network classifier to predict SST proficiency score (1-9) from
preprocessed speech transcripts using a Bag-of-Words representation.

In [ ]:
# Mount Google Drive to access the preprocessed corpus

from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix

from keras import models
from keras import layers
from keras.utils import np_utils
import tensorflow as tf

# ── Load preprocessed corpus ────────────────────────────
CSV_PATH = '/content/gdrive/MyDrive/NICT_JLE_4.1/Output/preprocessed_corpus.csv'

transcripts = []
y_output = []

with open(CSV_PATH, 'r', encoding='utf8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transcripts.append(row['transcript'])
        y_output.append(int(row['score']))

print(f"Loaded {len(transcripts)} transcripts.")
print(f"Score range: {min(y_output)} - {max(y_output)}")

We build a unified vocabulary from all candidate transcripts so the BoW
representation is as general as possible.

> Reference: [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
# ── Bag-of-Words vectorization ──────────────────────────
vectorizer = CountVectorizer()
vectorizer.fit(transcripts)

print(vectorizer.vocabulary_)
print(f"\nVocabulary size: {len(vectorizer.get_feature_names_out())}")

# Encode all documents
vector = vectorizer.transform(transcripts)

## Neural Network

We propose a **cone-shaped** architecture, decreasing the number of neurons per
layer as it approaches the output.

$$
(input) \cdot x \cdot x \cdot x \cdot x = (output)
$$

$$
x^4 = \frac{output}{input}
\qquad
\sqrt[4]{\frac{output}{input}} \approx 0.15
$$

In [ ]:
# ── Prepare data ────────────────────────────────────────
n = vector.shape[0]   # Number of candidates (1281)
m = vector.shape[1]   # Vocabulary size

X = vector.toarray()

y = np.array(y_output)
y = np_utils.to_categorical(y)
y = y[:, 1:10]        # Drop column 0 (unused, scores start at 1)

# ── Train / validation split (80 / 20) ──────────────────
N_val = round(0.2 * n)

X_val = X[:N_val]
y_val = y[:N_val]
X_train = X[N_val:]
y_train = y[N_val:]

# ── Build model ─────────────────────────────────────────
model = models.Sequential()
model.add(layers.Dense(2273, activation='sigmoid', input_shape=(m,)))
model.add(layers.Dense(341, activation='sigmoid'))
model.add(layers.Dense(51, activation='sigmoid'))
model.add(layers.Dense(9, activation='softmax'))

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ── Train ───────────────────────────────────────────────
history = model.fit(X_train,
                    y_train,
                    epochs=35,
                    batch_size=750,
                    validation_data=(X_val, y_val))

# ── Plot loss ───────────────────────────────────────────
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)

plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# ── Plot accuracy ───────────────────────────────────────
val_acc = history.history['val_accuracy']

plt.figure()
plt.plot(epochs, val_acc, 'r', label='Validation acc')
plt.title('Evolution of validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# ── Model summary ───────────────────────────────────────
model.summary()

## Performance Analysis

In [ ]:
# ── Compute accuracy ────────────────────────────────────
predictions = model.predict(X_val)
correct = 0

for i in range(len(X_val)):
    if y_val[i].argmax() == predictions[i].argmax():
        correct += 1

print(f"Final accuracy: {correct / len(X_val):.4f}")

> **References:**
> [Confusion matrix for multiple classes](https://stackoverflow.com/questions/65618137/confusion-matrix-for-multiple-classes-in-python)
> — [Seaborn](https://seaborn.pydata.org/index.html)

In [ ]:
# ── Confusion matrix ────────────────────────────────────
class_names = [1, 2, 3, 4, 5, 6, 7, 8, 9]
y_true_idx = np.argmax(y_val, axis=-1)
y_pred_idx = np.argmax(predictions, axis=-1)

cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(9))

fig = plt.figure(figsize=(16, 14))
ax = plt.subplot()
sns.heatmap(cm, annot=True, ax=ax, fmt='g')

ax.set_xlabel('Predicted', fontsize=20)
ax.xaxis.set_label_position('bottom')
ax.xaxis.set_ticklabels(class_names, fontsize=10)
ax.xaxis.tick_bottom()

ax.set_ylabel('True', fontsize=20)
ax.yaxis.set_ticklabels(class_names, fontsize=10)

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.title('Confusion Matrix — English Proficiency Prediction (NLP)', fontsize=20)

plt.savefig('confusion_matrix.png')
plt.show()